# 👤 NovaPay · Base de Datos de Clientes
### Construcción de perfiles de comportamiento para el panel del analista

In [1]:
import pandas as pd
from datetime import datetime, timedelta, timezone

In [3]:
df = pd.read_csv("C:/Users/Usuario/Desktop/the bridge/github/Desafio_Grupo1/data/synthetic_fin_data_CLEAN.csv")

In [4]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,hour_of_the_day,ip_country,merchant_category,balance_discrepancy
0,162,CASH_OUT,183806.32,C691771226,19391.00,0.00,C1416312719,382572.19,566378.51,0,18,US,restaurant,NaN
1,137,PAYMENT,521.37,C203378011,0.00,0.00,M42773300,0.00,0.00,0,17,FR,fuel,NaN
2,179,PAYMENT,3478.18,C1698571270,19853.00,16374.82,M643984524,0.00,0.00,0,11,ES,grocery,NaN
3,355,PAYMENT,1716.05,C913764937,5769.17,4053.13,M1387429131,0.00,0.00,0,19,GB,fuel,NaN
4,354,CASH_IN,253129.93,C2017736577,1328499.49,1581629.42,C407484102,2713220.48,2460090.55,0,18,DE,fuel,NaN


In [5]:
df.shape

(319636, 14)

In [6]:
customer_db = df.groupby("nameOrig").agg(
    total_transactions  = ("amount", "count"),
    total_amount        = ("amount", "sum"),     # Renombrado: de total_volume a total_amount
    fraud_flags         = ("isFraud", "sum"),    # Cambiado: extraemos la suma de fraudes reales (int)
    last_seen_step      = ("step", "max")        # Temporal: solo para calcular la fecha
).reset_index()

customer_db = customer_db.rename(columns={"nameOrig": "client_id"})

customer_db["risk_profile"] = customer_db["fraud_flags"].apply(
    lambda x: "high" if x > 0 else "low"
)

base_date = datetime(2024, 1, 1, tzinfo=timezone.utc)

customer_db["last_seen"] = customer_db["last_seen_step"].apply(
    lambda x: (base_date + timedelta(hours=x)).isoformat()
)

# 5. Inyectar 'createdAt' y 'updatedAt' para evitar el error de nulos
now_utc = datetime.now(timezone.utc).isoformat()
customer_db["createdAt"] = now_utc
customer_db["updatedAt"] = now_utc

# 6. Filtrar y ordenar estrictamente las columnas que existen en la BD
columnas_finales = [
    "client_id", 
    "total_transactions", 
    "total_amount", 
    "fraud_flags", 
    "last_seen", 
    "risk_profile", 
    "createdAt", 
    "updatedAt"
]
customer_db = customer_db[columnas_finales]

# 7. Comprobación y exportación final
print(customer_db.shape)
print(customer_db.head())

# Exportar a un nuevo JSON limpio
customer_db.to_json("customer_db_clean.json", orient="records")

(273195, 8)
     client_id  total_transactions  total_amount  fraud_flags  \
0  C1000013879                   1     516459.01            0   
1   C100001401                   1      17686.93            0   
2  C1000018432                   1      12216.58            0   
3  C1000036340                   1     253648.68            1   
4  C1000040770                   1      15486.87            0   

                   last_seen risk_profile                         createdAt  \
0  2024-01-09T00:00:00+00:00          low  2026-05-27T11:32:08.991925+00:00   
1  2024-01-13T14:00:00+00:00          low  2026-05-27T11:32:08.991925+00:00   
2  2024-01-06T18:00:00+00:00          low  2026-05-27T11:32:08.991925+00:00   
3  2024-01-28T07:00:00+00:00         high  2026-05-27T11:32:08.991925+00:00   
4  2024-01-08T17:00:00+00:00          low  2026-05-27T11:32:08.991925+00:00   

                          updatedAt  
0  2026-05-27T11:32:08.991925+00:00  
1  2026-05-27T11:32:08.991925+00:00  
2  2026-

In [7]:
print(customer_db.columns.tolist())
print(customer_db.dtypes)

['client_id', 'total_transactions', 'total_amount', 'fraud_flags', 'last_seen', 'risk_profile', 'createdAt', 'updatedAt']
client_id              object
total_transactions      int64
total_amount          float64
fraud_flags             int64
last_seen              object
risk_profile           object
createdAt              object
updatedAt              object
dtype: object
